# Pré-processamento

**Entrega: 07/11/2026 | Valor: 4,00 pontos.**

Este notebook é um **roteiro de trabalho sem implementação**. Cada um dos cinco membros tem uma seção própria para desenvolver. Substituir Membro 1–5 pelos nomes do grupo do CANVAS.


## Regras comuns às cinco tarefas

- Usar os CSVs oficiais em `data/`: `treino.csv`, `teste_sem_rotulo.csv` e `modelo_de_submissao.csv`.
- O alvo é `class`, com valores `normal` e `anomaly`; não remapear para cinco famílias de ataques.
- Não substituir, reembaralhar ou recriar a separação oficial de treino/teste. Validação interna é permitida **somente dentro do treino**.
- Ajustar imputação, codificação, normalização, balanceamento e redução apenas no treino de cada dobra.
- `id` identifica o teste e não é atributo preditivo. O teste não tem gabarito disponível à equipe.
- Investigar a associação de **cada atributo** com a classe antes de treinar modelos. Correlação alta não prova vazamento sozinha.
- Examinar também as colunas adicionadas, inclusive `score_ids_legado`, `tipo_alerta_soc` e `tempo_resposta_analista_min`; justificar decisões pela origem e disponibilidade da informação.
- Não reutilizar conclusões ou números da base antiga. Não procurar/reconstruir rótulos do teste público.
- Os módulos em `src/` são a estrutura legada e precisam ser adaptados pelos responsáveis; este roteiro não os executa.
- Cada membro deve produzir código comentado, resultados e uma conclusão autoral em sua seção, além de revisar a integração.


## Membro 1 — Ausentes, redundâncias e inconsistências

**Tarefa:** Implementar a carga comum e as estratégias indicadas pela etapa 1. Comparar tratamento de ausentes, normalização de categorias e manutenção/remoção de redundâncias com uma referência. Definir tratamento de duplicatas/conflitos sem contaminar validação e sem alterar o teste oficial.

**Entregável:** Transformações reutilizáveis e tabela de estratégias, evidências e justificativas. Nenhuma mediana ou decisão estatística aprendida fora das dobras.


### 1.1 Carga comum

Funções `carregar_treino()` e `carregar_teste()` para todos os membros. Os CSVs oficiais não são alterados, o alvo vira `y` (1 = `anomaly`), e `id` nunca entra como atributo.
Os sinais operacionais (`score_ids_legado`, `tipo_alerta_soc`, `tempo_resposta_analista_min`) ficam **fora** dos preditores por padrão, porque a etapa 1 indicou risco de vazamento (o tempo do analista existe sobretudo após a triagem). A decisão final é do Membro 5; basta passar `incluir_operacionais=True` para reavaliar.

In [1]:
# Membro 1 — 1.1 Carga comum (reutilizável pelos demais membros)
from pathlib import Path
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

SEMENTE = 42
N_DOBRAS = 5

project_root = Path.cwd().resolve()
if not (project_root / "data" / "treino.csv").exists():
    project_root = project_root.parent
if not (project_root / "data" / "treino.csv").exists():
    raise FileNotFoundError("Execute na raiz do projeto ou em notebooks/.")
DIR_DADOS = project_root / "data"

COL_ROTULO = "class"
COL_ID = "id"
CLASSES = ["normal", "anomaly"]          # anomaly = classe positiva (1)
COLS_CATEGORICAS = ["protocol_type", "service", "flag"]
# Sinais operacionais: ficam fora dos preditores até a conclusão do Membro 5 (etapa 1, seção 1.7).
SINAIS_OPERACIONAIS = ["score_ids_legado", "tipo_alerta_soc", "tempo_resposta_analista_min"]
# Colunas derivadas -> coluna de origem e relação esperada (verificadas na seção 1.2).
COLS_DERIVADAS = {
    "src_bytes_kb": ("src_bytes", "src_bytes / 1024"),
    "dst_bytes_kb": ("dst_bytes", "dst_bytes / 1024"),
    "total_bytes": ("src_bytes + dst_bytes", "src_bytes + dst_bytes"),
    "same_srv_pct": ("same_srv_rate", "same_srv_rate * 100"),
    "protocolo_cod": ("protocol_type", "código 1:1 de protocol_type"),
}
# Limites de domínio da documentação do KDD (conhecimento prévio, não aprendidos dos dados).
LIMITES_DOMINIO = {
    **{c: (0, 1) for c in ["serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
                           "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
                           "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
                           "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
                           "dst_host_serror_rate", "dst_host_srv_serror_rate",
                           "dst_host_rerror_rate", "dst_host_srv_rerror_rate"]},
    "same_srv_pct": (0, 100),
    "count": (0, 511), "srv_count": (0, 511),
    "dst_host_count": (0, 255), "dst_host_srv_count": (0, 255),
    "duration": (0, np.inf), "src_bytes": (0, np.inf), "dst_bytes": (0, np.inf),
    "total_bytes": (0, np.inf), "src_bytes_kb": (0, np.inf), "dst_bytes_kb": (0, np.inf),
}


def carregar_treino(incluir_operacionais: bool = False):
    """Lê treino.csv sem alterá-lo. Retorna (X, y) com y = 1 para anomaly."""
    df = pd.read_csv(DIR_DADOS / "treino.csv")
    assert set(df[COL_ROTULO]) == set(CLASSES) and df[COL_ROTULO].notna().all()
    assert COL_ID not in df.columns, "id não é atributo preditivo."
    descartar = [COL_ROTULO] + ([] if incluir_operacionais else SINAIS_OPERACIONAIS)
    X = df.drop(columns=descartar)
    y = (df[COL_ROTULO] == "anomaly").astype(int).rename(COL_ROTULO)
    return X, y


def carregar_teste(incluir_operacionais: bool = False):
    """Lê teste_sem_rotulo.csv somente para checar esquema/aplicar o pipeline já ajustado.

    Nenhuma estatística é calculada com o teste; ordem e IDs são preservados.
    """
    df = pd.read_csv(DIR_DADOS / "teste_sem_rotulo.csv")
    ids = df[COL_ID].copy()
    descartar = [COL_ID] + ([] if incluir_operacionais else SINAIS_OPERACIONAIS)
    return ids, df.drop(columns=descartar)


X_treino, y_treino = carregar_treino()
print(f"Treino: {X_treino.shape[0]:,} linhas x {X_treino.shape[1]} atributos candidatos "
      f"(sinais operacionais excluídos: {SINAIS_OPERACIONAIS})")
print("Distribuição do alvo:", y_treino.map({0: "normal", 1: "anomaly"}).value_counts().to_dict())

Treino: 69,833 linhas x 46 atributos candidatos (sinais operacionais excluídos: ['score_ids_legado', 'tipo_alerta_soc', 'tempo_resposta_analista_min'])
Distribuição do alvo: {'normal': 65754, 'anomaly': 4079}


### 1.2 Evidências de redundância e inconsistência

Antes de comparar estratégias, confirmamos com igualdades exatas (não com correlação) o que a etapa 1 apontou: colunas derivadas, constantes, variações de escrita e valores fora do domínio. São verificações estruturais, sem nenhum parâmetro aprendido para o modelo.

In [2]:
# Membro 1 — 1.2 Evidências estruturais: redundâncias, constantes, categorias, domínio
# Verificações determinísticas (igualdades e contagens). Não geram parâmetros usados pelo modelo.
def _rel(nome, original, derivada):
    ambos = original.notna() & derivada.notna()
    return {
        "derivada": nome,
        "relacao_testada": COLS_DERIVADAS[nome][1],
        "linhas_comparadas": int(ambos.sum()),
        "max_diferenca_abs": float((original[ambos] - derivada[ambos]).abs().max()),
        "mesmo_padrao_ausencia": bool(original.isna().equals(derivada.isna())),
        "ausentes_recuperaveis": int((original.isna() & derivada.notna()).sum()),
    }

Xe = X_treino
evid_redund = pd.DataFrame([
    _rel("src_bytes_kb", Xe["src_bytes"] / 1024, Xe["src_bytes_kb"]),
    _rel("dst_bytes_kb", Xe["dst_bytes"] / 1024, Xe["dst_bytes_kb"]),
    _rel("total_bytes", Xe["src_bytes"] + Xe["dst_bytes"], Xe["total_bytes"]),
    _rel("same_srv_pct", Xe["same_srv_rate"] * 100, Xe["same_srv_pct"]),
]).set_index("derivada")
# protocolo_cod: cada protocolo (normalizado) deve mapear para exatamente um código.
tab_proto = pd.crosstab(Xe["protocol_type"].str.strip().str.lower(), Xe["protocolo_cod"])
evid_redund.loc["protocolo_cod"] = {
    "relacao_testada": COLS_DERIVADAS["protocolo_cod"][1],
    "linhas_comparadas": int(Xe["protocol_type"].notna().sum()),
    "max_diferenca_abs": float(((tab_proto > 0).sum(axis=1) != 1).sum()),  # protocolos com >1 código
    "mesmo_padrao_ausencia": bool(Xe["protocol_type"].isna().equals(Xe["protocolo_cod"].isna())),
    "ausentes_recuperaveis": int((Xe["protocol_type"].isna() & Xe["protocolo_cod"].notna()).sum()),
}
print("Relações determinísticas (max_diferenca_abs = 0 => derivada redundante):")
display(evid_redund)
print("protocol_type normalizado x protocolo_cod:")
display(tab_proto)

constantes = [c for c in Xe.columns if Xe[c].nunique(dropna=True) <= 1]
print("Atributos constantes no treino completo:", constantes)

evid_cat = pd.DataFrame({
    "categorias_brutas": Xe[COLS_CATEGORICAS].nunique(),
    "categorias_normalizadas": [Xe[c].str.strip().str.lower().nunique() for c in COLS_CATEGORICAS],
    "ausentes": Xe[COLS_CATEGORICAS].isna().sum(),
})
print("Inconsistência de escrita nos categóricos (caixa/espaços):")
display(evid_cat)

fora = pd.DataFrame({c: (Xe[c] < lo) | (Xe[c] > hi) for c, (lo, hi) in LIMITES_DOMINIO.items()})
print(f"Células fora do domínio: {int(fora.values.sum()):,}; linhas afetadas: {int(fora.any(axis=1).sum()):,}")

Relações determinísticas (max_diferenca_abs = 0 => derivada redundante):


,relacao_testada,linhas_comparadas,max_diferenca_abs,mesmo_padrao_ausencia,ausentes_recuperaveis
derivada,,,,,
src_bytes_kb,src_bytes / 1024,41927,0.000000e+00,True,0
dst_bytes_kb,dst_bytes / 1024,69833,0.000000e+00,True,0
total_bytes,src_bytes + dst_bytes,41927,0.000000e+00,True,0
same_srv_pct,same_srv_rate * 100,66262,3.552714e-15,True,0
protocolo_cod,código 1:1 de protocol_type,69833,0.000000e+00,True,0


protocol_type normalizado x protocolo_cod:


protocolo_cod,0,1,2
protocol_type,,,
icmp,0,0,1643
tcp,56010,0,0
udp,0,12180,0


Atributos constantes no treino completo: ['num_outbound_cmds']
Inconsistência de escrita nos categóricos (caixa/espaços):


,categorias_brutas,categorias_normalizadas,ausentes
protocol_type,6,3,0
service,96,67,4416
flag,19,11,4093


Células fora do domínio: 6,121; linhas afetadas: 5,650


### 1.3 Transformações reutilizáveis

Cada etapa é um transformador do scikit-learn, encaixado em um `Pipeline`. Assim, medianas, modas, categorias do One-Hot e constantes são aprendidas **apenas no `fit`**, sempre chamado com o treino da dobra.

| Transformação | Aprende dos dados? | Opções comparadas |
|---|---|---|
| `NormalizarCategorias` | Não (strip + minúsculas) | brutas × normalizadas |
| `TratarForaDominio` | Não (limites da documentação) | manter × virar ausente × recortar |
| `RemoverRedundantes` | Sim: constantes da dobra | manter × remover derivadas + constantes |
| `SimpleImputer` numérico | Sim: mediana da dobra | mediana × mediana + indicador de ausência |
| `SimpleImputer` categórico | Sim: moda da dobra (se usada) | categoria `__ausente__` × moda |
| `filtrar_linhas_treino` | Sim: grupos do treino da dobra | todas × sem duplicatas × sem conflitos × ambos |

O One-Hot com `handle_unknown="ignore"` é só o mínimo para o modelo rodar. A comparação de codificações e escalas fica com o Membro 3.

In [3]:
# Membro 1 — 1.3 Transformações reutilizáveis (compatíveis com Pipeline do scikit-learn)
# Regra: tudo que aprende algo dos dados (constantes, medianas, modas, categorias) só aprende no fit,
# que é chamado apenas com o treino de cada dobra. O teste oficial só passa por transform.

class NormalizarCategorias(BaseEstimator, TransformerMixin):
    """Remove espaços e uniformiza caixa ('TCP ' -> 'tcp'). Sem estado; NaN continua NaN."""

    def __init__(self, colunas=tuple(COLS_CATEGORICAS), ativo=True):
        self.colunas = colunas
        self.ativo = ativo

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.colunas:
            if c in X:
                serie = X[c].astype(object)
                X[c] = serie.str.strip().str.lower() if self.ativo else serie
        return X


class TratarForaDominio(BaseEstimator, TransformerMixin):
    """Valores impossíveis pelos limites documentados.

    modo='manter' (referência), 'ausente' (vira NaN e segue para a imputação)
    ou 'recorte' (clip nos limites). Os limites vêm da documentação, não dos dados.
    """

    def __init__(self, limites=None, modo="manter"):
        self.limites = limites
        self.modo = modo

    def fit(self, X, y=None):
        assert self.modo in {"manter", "ausente", "recorte"}
        return self

    def transform(self, X):
        if self.modo == "manter":
            return X
        X = X.copy()
        for c, (lo, hi) in (self.limites or LIMITES_DOMINIO).items():
            if c not in X:
                continue
            if self.modo == "ausente":
                X[c] = X[c].astype(float).mask((X[c] < lo) | (X[c] > hi))
            else:
                X[c] = X[c].clip(lower=lo, upper=hi)
        return X


class RemoverRedundantes(BaseEstimator, TransformerMixin):
    """Remove derivadas determinísticas (lista fixa) e constantes detectadas NO TREINO DA DOBRA."""

    def __init__(self, derivadas=tuple(COLS_DERIVADAS), ativo=True):
        self.derivadas = derivadas
        self.ativo = ativo

    def fit(self, X, y=None):
        if not self.ativo:
            self.remover_ = []
            return self
        constantes = [c for c in X.columns if X[c].nunique(dropna=True) <= 1]
        self.constantes_ = constantes
        self.remover_ = sorted(set(self.derivadas).intersection(X.columns) | set(constantes))
        return self

    def transform(self, X):
        return X.drop(columns=self.remover_)


def _eh_categorica(X):
    return [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]


def _eh_numerica(X):
    return [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]


def montar_preprocessamento(normalizar_categorias=True, fora_dominio="manter",
                            remover_redundantes=False, imputacao_num="mediana",
                            imputacao_cat="constante"):
    """Bloco de pré-processamento do Membro 1 (+ One-Hot mínimo, provisório até o Membro 3).

    imputacao_num: 'mediana' | 'mediana+indicador'
    imputacao_cat: 'constante' (ausência vira a categoria '__ausente__') | 'moda'
    """
    imp_num = SimpleImputer(strategy="median",
                            add_indicator=(imputacao_num == "mediana+indicador"))
    imp_cat = (SimpleImputer(strategy="constant", fill_value="__ausente__")
               if imputacao_cat == "constante" else SimpleImputer(strategy="most_frequent"))
    colunas = ColumnTransformer([
        ("num", imp_num, _eh_numerica),
        ("cat", Pipeline([("imp", imp_cat),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
         _eh_categorica),
    ], verbose_feature_names_out=False)
    return Pipeline([
        ("categorias", NormalizarCategorias(ativo=normalizar_categorias)),
        ("dominio", TratarForaDominio(modo=fora_dominio)),
        ("redundantes", RemoverRedundantes(ativo=remover_redundantes)),
        ("imputacao_codificacao", colunas),
    ])


def chave_registro(X):
    """Identificador de registros com atributos idênticos (categorias normalizadas, NaN como valor)."""
    Xn = NormalizarCategorias().transform(X)
    return pd.Series(pd.factorize(pd.util.hash_pandas_object(Xn, index=False))[0], index=X.index)


def filtrar_linhas_treino(X, y, estrategia="todas"):
    """Duplicatas/conflitos tratados SOMENTE no treino da dobra; a validação fica intacta.

    'todas'                      : mantém tudo (referência)
    'sem_duplicatas'             : remove cópias com mesmos atributos e mesma classe (mantém a 1ª)
    'sem_conflitos'              : remove TODOS os registros de grupos com atributos iguais e
                                   classes divergentes (não inventa nem troca rótulos)
    'sem_duplicatas_e_conflitos' : as duas remoções
    """
    if estrategia == "todas":
        return X, y
    if estrategia not in {"sem_duplicatas", "sem_conflitos", "sem_duplicatas_e_conflitos"}:
        raise ValueError(estrategia)
    chave = chave_registro(X)
    manter = pd.Series(True, index=X.index)
    if "duplicatas" in estrategia:
        manter &= ~pd.DataFrame({"k": chave, "y": y}).duplicated()
    if "conflitos" in estrategia:
        manter &= ~chave.map(y.groupby(chave).nunique()).gt(1)
    return X[manter], y[manter]

### 1.4 Dobras sem contaminação por duplicatas

Com `StratifiedKFold` comum, cópias idênticas de um registro caem em dobras diferentes, e a validação passa a medir memorização. Usamos `StratifiedGroupKFold`, com grupo = registro de atributos idênticos (categorias normalizadas, `NaN` tratado como valor). A auditoria abaixo compara as duas divisões. `INDICES_DOBRAS` fica disponível para os demais membros usarem exatamente as mesmas dobras.

In [4]:
# Membro 1 — 1.4 Dobras sem contaminação por cópias
# Registros com atributos idênticos recebem o mesmo grupo e ficam sempre na mesma dobra.
grupos_treino = chave_registro(X_treino)


def auditar_dobras(divisor, nome, usa_grupos):
    linhas = []
    args = (X_treino, y_treino, grupos_treino) if usa_grupos else (X_treino, y_treino)
    for i, (itr, iva) in enumerate(divisor.split(*args), start=1):
        chaves_tr = set(grupos_treino.iloc[itr])
        copia_no_treino = grupos_treino.iloc[iva].isin(chaves_tr)
        linhas.append({"divisao": nome, "dobra": i, "n_validacao": len(iva),
                       "pct_anomaly_validacao": 100 * y_treino.iloc[iva].mean(),
                       "pct_validacao_com_copia_no_treino": 100 * copia_no_treino.mean()})
    return linhas


DOBRAS = StratifiedGroupKFold(n_splits=N_DOBRAS, shuffle=True, random_state=SEMENTE)
auditoria = pd.DataFrame(
    auditar_dobras(StratifiedKFold(N_DOBRAS, shuffle=True, random_state=SEMENTE), "StratifiedKFold", False)
    + auditar_dobras(DOBRAS, "StratifiedGroupKFold (grupos = registro)", True)
)
print(f"Grupos de registros distintos: {grupos_treino.nunique():,} para {len(grupos_treino):,} linhas")
display(auditoria.groupby("divisao")[["n_validacao", "pct_anomaly_validacao",
                                      "pct_validacao_com_copia_no_treino"]].agg(["mean", "min", "max"]))
# Estrutura dos conflitos (atributos idênticos, classes diferentes) no treino completo — só diagnóstico.
perfil_grupos = y_treino.groupby(grupos_treino).agg(["size", "sum", "nunique"])
conflitos = perfil_grupos[perfil_grupos["nunique"] > 1]
print(f"Grupos conflitantes: {len(conflitos):,}; linhas: {int(conflitos['size'].sum()):,}; "
      f"anomalias envolvidas: {int(conflitos['sum'].sum()):,} de {int(y_treino.sum()):,} "
      f"({100 * conflitos['sum'].sum() / y_treino.sum():.1f}%)")
print("Composição dos grupos conflitantes (linhas x anomalias no grupo):")
display(pd.crosstab(conflitos["size"].rename("linhas_no_grupo"), conflitos["sum"].rename("anomalias_no_grupo")))

# Índices fixos das dobras, para os demais membros reutilizarem exatamente as mesmas divisões.
INDICES_DOBRAS = list(DOBRAS.split(X_treino, y_treino, grupos_treino))

Grupos de registros distintos: 56,766 para 69,833 linhas


n_validacao                \
                                                mean    min    max   
divisao                                                              
StratifiedGroupKFold (grupos = registro)     13966.6  13966  13968   
StratifiedKFold                              13966.6  13966  13967   

                                         pct_anomaly_validacao            \
                                                          mean       min   
divisao                                                                    
StratifiedGroupKFold (grupos = registro)              5.841078  5.835601   
StratifiedKFold                                       5.841078  5.835601   

                                                    \
                                               max   
divisao                                              
StratifiedGroupKFold (grupos = registro)  5.842761   
StratifiedKFold                           5.842761   

                                         pct_validacao_com_copia_no_treino  \
                                                                      mean   
divisao                                                                      
StratifiedGroupKFold (grupos = registro)                          0.000000   
StratifiedKFold                                                  29.709452   

                                                                
                                                min        max  
divisao                                                         
StratifiedGroupKFold (grupos = registro)   0.000000   0.000000  
StratifiedKFold                           29.505262  29.956326

Grupos conflitantes: 1,702; linhas: 3,730; anomalias envolvidas: 1,896 de 4,079 (46.5%)
Composição dos grupos conflitantes (linhas x anomalias no grupo):


anomalias_no_grupo,1,2
linhas_no_grupo,,
2,1376,0
3,132,194


### 1.5 Comparação com a referência (um fator por vez)

- **R0 (referência):** categorias brutas, valores fora do domínio mantidos, mediana nos numéricos, categoria `__ausente__` nos categóricos, todas as colunas e todas as linhas.
- Cada variação altera **um único fator** em relação a R0.
- Modelo provisório: `DecisionTreeClassifier(random_state=42)`. Árvores não dependem de escala, o que isola os efeitos desta seção do trabalho do Membro 3.
- Métrica principal: F1-macro na validação de cada dobra. A validação **nunca** é filtrada (duplicatas e conflitos continuam nela), então a estimativa reflete dados com o mesmo ruído.

In [5]:
# Membro 1 — 1.5 Comparação das estratégias (um fator por vez contra a referência)
# Modelo de referência provisório: árvore de decisão (insensível a escala, isola os efeitos do Membro 1).
# Mesmo modelo, mesma semente e mesmas dobras para todas as estratégias.
MODELO_REFERENCIA = DecisionTreeClassifier(random_state=SEMENTE)

REFERENCIA = dict(normalizar_categorias=False, fora_dominio="manter", remover_redundantes=False,
                  imputacao_num="mediana", imputacao_cat="constante", linhas_treino="todas")
VARIACOES = {
    "R0 referência": ("—", {}),
    "A1 categorias normalizadas": ("categorias", {"normalizar_categorias": True}),
    "B1 categórico: moda": ("ausentes categóricos", {"imputacao_cat": "moda"}),
    "B2 numérico: mediana + indicador": ("ausentes numéricos", {"imputacao_num": "mediana+indicador"}),
    "C1 fora do domínio -> ausente": ("inconsistências de domínio", {"fora_dominio": "ausente"}),
    "C2 fora do domínio -> recorte": ("inconsistências de domínio", {"fora_dominio": "recorte"}),
    "D1 remover derivadas e constantes": ("redundância de atributos", {"remover_redundantes": True}),
    "E1 treino sem duplicatas": ("redundância de registros", {"linhas_treino": "sem_duplicatas"}),
    "E2 treino sem conflitos": ("redundância de registros", {"linhas_treino": "sem_conflitos"}),
    "E3 treino sem duplicatas e conflitos": ("redundância de registros",
                                             {"linhas_treino": "sem_duplicatas_e_conflitos"}),
}


def avaliar_estrategia(config, modelo=MODELO_REFERENCIA, dobras=INDICES_DOBRAS):
    """Ajusta pré-processamento + modelo só no treino de cada dobra e avalia na validação intacta."""
    config = dict(config)
    linhas_treino = config.pop("linhas_treino")
    resultados = []
    for i, (itr, iva) in enumerate(dobras, start=1):
        X_tr, y_tr = filtrar_linhas_treino(X_treino.iloc[itr], y_treino.iloc[itr], linhas_treino)
        X_va, y_va = X_treino.iloc[iva], y_treino.iloc[iva]
        pipe = Pipeline([("prep", montar_preprocessamento(**config)), ("modelo", clone(modelo))])
        t0 = time.perf_counter()
        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_va)
        f1_normal, f1_anomaly = f1_score(y_va, pred, average=None, labels=[0, 1])
        resultados.append({
            "dobra": i, "f1_macro": (f1_normal + f1_anomaly) / 2,
            "f1_normal": f1_normal, "f1_anomaly": f1_anomaly,
            "n_linhas_treino": len(X_tr), "n_atributos_finais": pipe[:-1].transform(X_va.head(1)).shape[1],
            "segundos": time.perf_counter() - t0,
        })
    return pd.DataFrame(resultados)


por_dobra = []
for nome, (fator, mudanca) in VARIACOES.items():
    res = avaliar_estrategia({**REFERENCIA, **mudanca})
    res.insert(0, "estrategia", nome)
    res.insert(1, "fator", fator)
    por_dobra.append(res)
por_dobra_m1 = pd.concat(por_dobra, ignore_index=True)

ref_f1 = por_dobra_m1.loc[por_dobra_m1.estrategia == "R0 referência"].set_index("dobra")["f1_macro"]
por_dobra_m1["delta_vs_R0"] = por_dobra_m1["f1_macro"] - por_dobra_m1["dobra"].map(ref_f1)

tabela_m1 = por_dobra_m1.groupby(["estrategia", "fator"], sort=False).agg(
    f1_macro_media=("f1_macro", "mean"), f1_macro_dp=("f1_macro", "std"),
    f1_anomaly=("f1_anomaly", "mean"), f1_normal=("f1_normal", "mean"),
    delta_vs_R0=("delta_vs_R0", "mean"),
    dobras_melhores_que_R0=("delta_vs_R0", lambda d: int((d > 0).sum())),
    linhas_treino=("n_linhas_treino", "mean"), atributos=("n_atributos_finais", "mean"),
    segundos=("segundos", "mean"),
).reset_index()
with pd.option_context("display.float_format", "{:.4f}".format):
    display(tabela_m1)
    print("F1-macro por dobra:")
    display(por_dobra_m1.pivot(index="estrategia", columns="dobra", values="f1_macro")
            .reindex(VARIACOES))

,estrategia,fator,f1_macro_media,f1_macro_dp,f1_anomaly,f1_normal,delta_vs_R0,dobras_melhores_que_R0,linhas_treino,atributos,segundos
0,R0 referência,—,0.8049,0.0054,0.6297,0.9800,0.0000,0,55866.4000,162.0000,11.9532
1,A1 categorias normalizadas,categorias,0.8027,0.0072,0.6255,0.9799,-0.0021,1,55866.4000,125.2000,9.4967
2,B1 categórico: moda,ausentes categóricos,0.8055,0.0082,0.6311,0.9800,0.0007,2,55866.4000,160.0000,11.7405
3,B2 numérico: mediana + indicador,ausentes numéricos,0.8048,0.0035,0.6296,0.9800,-0.0001,2,55866.4000,187.0000,14.8842
4,C1 fora do domínio -> ausente,inconsistências de domínio,0.8051,0.0043,0.6302,0.9800,0.0003,3,55866.4000,162.0000,13.3021
5,C2 fora do domínio -> recorte,inconsistências de domínio,0.8040,0.0063,0.6280,0.9799,-0.0009,1,55866.4000,162.0000,4.3843
6,D1 remover derivadas e constantes,redundância de atributos,0.8055,0.0056,0.6310,0.9800,0.0006,2,55866.4000,155.8000,2.5893
7,E1 treino sem duplicatas,redundância de registros,0.7817,0.0094,0.5847,0.9787,-0.0232,0,46774.4000,162.0000,7.4173
8,E2 treino sem conflitos,redundância de registros,0.8717,0.0041,0.7587,0.9847,0.0668,5,52882.4000,157.4000,7.6126
9,E3 treino sem duplicatas e conflitos,redundância de registros,0.8698,0.0046,0.7552,0.9845,0.0650,5,44051.2000,157.4000,5.8497


F1-macro por dobra:


dobra,1,2,3,4,5
estrategia,,,,,
R0 referência,0.8059,0.8106,0.8084,0.8028,0.7966
A1 categorias normalizadas,0.7963,0.8036,0.8074,0.8116,0.7948
B1 categórico: moda,0.7965,0.8092,0.8179,0.8010,0.8032
B2 numérico: mediana + indicador,0.8012,0.8089,0.8022,0.8036,0.8082
C1 fora do domínio -> ausente,0.8123,0.8030,0.8009,0.8050,0.8045
C2 fora do domínio -> recorte,0.8017,0.8044,0.8051,0.8131,0.7957
D1 remover derivadas e constantes,0.7957,0.8099,0.8069,0.8076,0.8073
E1 treino sem duplicatas,0.7662,0.7839,0.7860,0.7811,0.7912
E2 treino sem conflitos,0.8702,0.8724,0.8784,0.8681,0.8693


### 1.6 Combinação proposta e compatibilidade com o teste

Juntamos as opções escolhidas (justificadas na tabela abaixo) e conferimos que o pipeline ajustado **somente no treino** transforma o teste oficial sem remover nem reordenar linhas e sem deixar ausentes. Nenhuma estatística do teste é calculada e nenhuma previsão é gerada.

In [6]:
# Membro 1 — 1.6 Combinação proposta e checagem de compatibilidade com o teste oficial
# Opções escolhidas pelos resultados da 1.5 e por justificativa técnica (ver tabela 1.7).
# Atenção: escolher e reavaliar nas mesmas dobras é otimista; o Membro 5 deve confirmar na integração.
CONFIG_M1 = dict(normalizar_categorias=True, fora_dominio="ausente", remover_redundantes=True,
                 imputacao_num="mediana", imputacao_cat="constante", linhas_treino="sem_conflitos")

res_comb = avaliar_estrategia(CONFIG_M1)
res_comb.insert(0, "estrategia", "F1 combinação proposta")
res_comb["delta_vs_R0"] = res_comb["f1_macro"] - res_comb["dobra"].map(ref_f1)
with pd.option_context("display.float_format", "{:.4f}".format):
    display(res_comb)
print(f"F1-macro combinação: {res_comb.f1_macro.mean():.4f} ± {res_comb.f1_macro.std():.4f} "
      f"(R0: {ref_f1.mean():.4f}); melhor que R0 em {(res_comb.delta_vs_R0 > 0).sum()}/{N_DOBRAS} dobras")

# Compatibilidade: ajusta SÓ no treino e transforma o teste sem remover/reordenar linhas.
# Nenhuma estatística do teste é usada; nenhuma previsão é gerada aqui.
config_prep = {k: v for k, v in CONFIG_M1.items() if k != "linhas_treino"}
X_fit, y_fit = filtrar_linhas_treino(X_treino, y_treino, CONFIG_M1["linhas_treino"])
prep_m1 = montar_preprocessamento(**config_prep).fit(X_fit, y_fit)
ids_teste, X_teste = carregar_teste()
assert list(X_teste.columns) == list(X_treino.columns), "Esquema do teste difere do treino."
Z_teste = prep_m1.transform(X_teste)
assert Z_teste.shape[0] == len(ids_teste) and not np.isnan(Z_teste).any()
print(f"Teste transformado: {Z_teste.shape[0]:,} linhas (ordem e IDs preservados), "
      f"{Z_teste.shape[1]} colunas, sem ausentes.")
print("Colunas removidas no ajuste:", prep_m1.named_steps["redundantes"].remover_)

,estrategia,dobra,f1_macro,f1_normal,f1_anomaly,n_linhas_treino,n_atributos_finais,segundos,delta_vs_R0
0,F1 combinação proposta,1,0.8685,0.9841,0.7528,52884,117,5.3332,0.0626
1,F1 combinação proposta,2,0.8703,0.9845,0.7561,52879,117,5.0844,0.0597
2,F1 combinação proposta,3,0.8779,0.9853,0.7705,52864,117,6.4827,0.0695
3,F1 combinação proposta,4,0.8664,0.9843,0.7485,52891,117,6.5390,0.0636
4,F1 combinação proposta,5,0.8701,0.9845,0.7557,52894,117,5.3297,0.0734


F1-macro combinação: 0.8706 ± 0.0044 (R0: 0.8049); melhor que R0 em 5/5 dobras


Teste transformado: 24,333 linhas (ordem e IDs preservados), 118 colunas, sem ausentes.
Colunas removidas no ajuste: ['dst_bytes_kb', 'num_outbound_cmds', 'protocolo_cod', 'same_srv_pct', 'src_bytes_kb', 'total_bytes']


### Resultados e justificativa — Membro 1

**Evidências estruturais (1.2).** As cinco colunas derivadas reproduzem exatamente a coluna de origem: `src_bytes_kb` = `src_bytes`/1024, `dst_bytes_kb` = `dst_bytes`/1024, `total_bytes` = `src_bytes` + `dst_bytes`, `same_srv_pct` = 100 × `same_srv_rate` (diferença máxima 4e-15, erro de ponto flutuante) e `protocolo_cod` 1:1 com `protocol_type` normalizado. Todas têm **o mesmo padrão de ausência** da origem, então não recuperam nenhum ausente. São redundância pura. `num_outbound_cmds` é constante. Os categóricos caem de 6→3, 96→67 e 19→11 categorias após normalizar caixa e espaços. Há 6.121 células fora do domínio, em 5.650 linhas.

**Duplicatas e conflitos (1.4).** As 69.833 linhas formam 56.766 registros distintos. Com `StratifiedKFold`, **~29,7% das linhas de validação têm uma cópia idêntica no treino da dobra**; com `StratifiedGroupKFold` agrupado por registro, 0%, mantendo 5,84% de `anomaly` em todas as dobras. Os conflitos são o achado mais relevante. São 1.702 grupos (3.730 linhas) com atributos idênticos e classes opostas, quase todos pares 1 normal + 1 anomaly. Eles concentram **1.896 das 4.079 anomalias (46,5%)**. O padrão sugere ruído de rótulo (cópias com rótulo trocado), mas não temos como saber qual rótulo é o verdadeiro.

**Comparação (1.5), F1-macro médio em 5 dobras (R0 = 0,8049 ± 0,0054):**

| Fator | Estratégias comparadas | Evidência (Δ F1-macro vs R0; dobras melhores) | Decisão e justificativa |
|---|---|---|---|
| Categorias inconsistentes | brutas × normalizadas | −0,0021; 1/5 (dentro do ruído, dp ≈ 0,005–0,007) | **Normalizar.** Desempenho equivalente com 162 → 125 colunas; sem isso, `TCP` e `tcp` seriam categorias diferentes, e uma variação nova no teste viraria "desconhecida". |
| Ausentes categóricos | `__ausente__` × moda | +0,0007; 2/5 (empate) | **Categoria `__ausente__`.** Empate no desempenho; mantém a ausência visível em vez de inventar o valor mais comum. |
| Ausentes numéricos | mediana × mediana + indicador | −0,0001; 2/5 (empate) | **Mediana sem indicador** para a árvore (+25 colunas sem ganho). Como `dst_host_serror_rate` tem ausência informativa (etapa 1), reavaliar o indicador com modelos lineares/MLP. |
| Valores fora do domínio | manter × ausente × recorte | ausente +0,0003 (3/5); recorte −0,0009 (1/5) | **Tratar como ausente.** Pequena vantagem e coerência: um valor impossível é erro de registro, e recortar inventaria um valor no limite. |
| Redundância de atributos | manter × remover derivadas + constantes | +0,0006; 2/5 | **Remover.** Igualdade exata comprovada, nenhum ausente recuperável; reduz 6 colunas sem perda. Constantes são detectadas no treino de cada dobra. |
| Duplicatas no treino | manter × remover | −0,0232; 0/5 | **Manter.** Remover cópias muda o peso das amostras e piora todas as dobras; a contaminação já é evitada pelo agrupamento das dobras. |
| Conflitos no treino | manter × remover grupos conflitantes | **+0,0668; 5/5** (0,8717 ± 0,0041; F1 anomaly 0,630 → 0,759) | **Remover só do treino de cada dobra**, sem trocar rótulos. A validação continua com os conflitos, então o ganho não vem de facilitar a avaliação. Remover também as duplicatas (E3) não acrescenta nada (0,8698). |

**Combinação proposta (1.6):** categorias normalizadas, fora do domínio → ausente, remoção de derivadas e constantes, mediana + `__ausente__` e treino sem conflitos. Resultado: **F1-macro 0,8706 ± 0,0044**, melhor que R0 nas 5 dobras, com 117 colunas (R0: 162). O ganho vem quase todo dos conflitos; os demais fatores são neutros em desempenho e foram escolhidos por coerência e menor dimensão. O pipeline ajustado só no treino transforma as 24.333 linhas do teste sem perdas, sem reordenação e sem ausentes.

**Limitações e pendências para a integração (Membro 5).**
- As opções foram escolhidas e reavaliadas nas mesmas dobras, o que torna a estimativa da combinação um pouco otimista. Confirmar com outra semente de dobras e com os modelos da etapa 3.
- O efeito dos conflitos foi medido com uma árvore; conferir se persiste com os demais modelos e com o balanceamento do Membro 2.
- O agrupamento considera iguais apenas registros idênticos, inclusive nos `NaN`. Cópias que diferem só por um valor ausente não são agrupadas.
- Os sinais operacionais estão excluídos até a conclusão do Membro 5.
- Nenhuma decisão usou o teste; o teste oficial não foi alterado.

## Membro 2 — Duas técnicas de balanceamento

**Tarefa:** Escolher e implementar pelo menos duas técnicas, comparando-as com nenhum balanceamento nas mesmas dobras e com o mesmo modelo de referência. Exemplos: SMOTE versus RandomUnderSampler, ou RandomOverSampler versus RandomUnderSampler. Justificar o tratamento de categorias caso use método sintético.

**Entregável:** Comparação de F1-macro e métricas por classe, distribuição antes/depois apenas no treino de cada dobra e discussão de perda de informação/sobreajuste.


In [7]:
# TODO — Membro 2: Duas técnicas de balanceamento
# Desenvolver a implementação autoral nesta célula.


### Resultados e justificativa — Membro 2

*A preencher pelo responsável após implementar e executar sua tarefa.*


## Membro 3 — Codificação e transformação numérica

**Tarefa:** Implementar pelo menos uma conversão simbólico-numérica e uma transformação numérica. Justificar One-Hot ou alternativa para service e tratamento de categorias desconhecidas. Comparar com uma referência; por exemplo, sem escala versus StandardScaler/RobustScaler.

**Entregável:** Blocos de codificação/escala ajustados por dobra, dimensão resultante e comparação explícita com justificativa técnica.


In [8]:
# TODO — Membro 3: Codificação e transformação numérica
# Desenvolver a implementação autoral nesta célula.


### Resultados e justificativa — Membro 3

*A preencher pelo responsável após implementar e executar sua tarefa.*


## Membro 4 — Redução de dimensionalidade ou seleção de instâncias

**Tarefa:** Escolher e implementar uma técnica que atenda literalmente a uma das duas alternativas do enunciado, por exemplo PCA. Comparar com/sem a técnica e justificar o parâmetro escolhido apenas por validação interna. Seleção de atributos isolada não substitui automaticamente a exigência.

**Entregável:** Comparação do desempenho, custo e dimensionalidade ou quantidade de instâncias; transformação reutilizável ajustada dentro das dobras.


In [9]:
# TODO — Membro 4: Redução de dimensionalidade ou seleção de instâncias
# Desenvolver a implementação autoral nesta célula.


### Resultados e justificativa — Membro 4

*A preencher pelo responsável após implementar e executar sua tarefa.*


## Membro 5 — Integração e comparação das estratégias

**Tarefa:** Combinar os blocos dos membros 1–4 em pipeline sem vazamento. Definir dobras internas comuns e modelo de referência com a equipe; executar comparações explícitas, preferencialmente alterando um fator por vez. Auditar duplicatas entre dobras e consolidar os resultados.

**Entregável:** Tabela estratégia × modelo de referência × métricas, resultados por dobra, discussão das escolhas e configurações para a etapa 3. Não apresentar só uma combinação final.


In [10]:
# TODO — Membro 5: Integração e comparação das estratégias
# Desenvolver a implementação autoral nesta célula.


### Resultados e justificativa — Membro 5

*A preencher pelo responsável após implementar e executar sua tarefa.*


## Revisão conjunta

Cada membro deve explicar sua contribuição e revisar a integração com os demais. Conferir os requisitos da etapa, resultados reais, ausência de vazamento, reprodução e declaração de uso de IA antes de entregar.
